# HotpotQA Dataset Explorer

This notebook loads the HotpotQA `fullwiki` split and helps you:
- Understand the dataset schema
- Inspect sample questions, answers, and supporting facts
- See exactly how EM and F1 are computed in this project

## 1. Load the Dataset

In [ ]:
from datasets import load_dataset

# Load 10 samples from the train split — same config used by run_all_systems.py
ds = load_dataset("hotpot_qa", "fullwiki", split="train[:10]", trust_remote_code=True)
items = list(ds)
print(f"Loaded {len(items)} items")

## 2. Dataset Schema

Print the keys and types of a single item so you know what fields are available.

In [ ]:
item = items[0]

print("Top-level keys:")
for key, val in item.items():
    print(f"  {key!r:25s}  type={type(val).__name__}")

print("\nDataset features schema:")
print(ds.features)

## 3. Question & Answer

Every item has a `question` string and a short `answer` string.

In [ ]:
for i, item in enumerate(items[:5]):
    print(f"--- Item {i} ---")
    print(f"  ID      : {item['id']}")
    print(f"  Question: {item['question']}")
    print(f"  Answer  : {item['answer']}")
    print(f"  Type    : {item.get('type', 'n/a')}")
    print()

## 4. Supporting Facts

HotpotQA is a *multi-hop* dataset — each question needs **two** Wikipedia pages to answer.
The `supporting_facts` field tells you which pages (and which sentences) contain the answer.

This is what makes HotpotQA hard: a single-page retrieval system can miss it.

In [ ]:
for i, item in enumerate(items[:3]):
    print(f"--- Item {i} ---")
    print(f"  Question        : {item['question']}")
    print(f"  Answer          : {item['answer']}")
    sf = item['supporting_facts']
    # supporting_facts is a dict with 'title' and 'sent_id' lists
    pages_needed = list(dict.fromkeys(sf['title']))  # deduplicated, order preserved
    print(f"  Pages needed    : {pages_needed}")
    print(f"  Sentence indices: {sf['sent_id']}")
    print()

## 5. Context Field

Each item also includes a `context` field — a list of `[title, sentences]` pairs.
This is the distractor context HotpotQA provides; our project ignores it and fetches Wikipedia directly.

In [ ]:
item = items[0]
ctx = item['context']  # dict with 'title' list and 'sentences' list-of-lists

print(f"Question : {item['question']}")
print(f"Answer   : {item['answer']}")
print(f"\nContext passages ({len(ctx['title'])} pages):")
for title, sents in zip(ctx['title'], ctx['sentences']):
    preview = ' '.join(sents[:2])[:120]
    print(f"  [{title}]  {preview}...")

## 6. How EM is Computed in This Project

**Exact Match (retrieval):** does the gold answer string appear (case-insensitive) anywhere in the retrieved context?

This is a *retrieval* EM, not a generation EM — there is no LLM involved in M2.

In [ ]:
def compute_em(prediction: str, gold: str) -> float:
    """1.0 if gold appears anywhere in prediction (case-insensitive)."""
    return 1.0 if gold.lower() in prediction.lower() else 0.0

# Example 1: gold answer IS in the context
context_good = "France is a country in Western Europe. The capital of France is Paris."
gold = "Paris"
print(f"Context : {context_good}")
print(f"Gold    : {gold!r}")
print(f"EM      : {compute_em(context_good, gold)}")

print()

# Example 2: gold answer NOT in the context
context_bad = "France has a long history of art and culture."
print(f"Context : {context_bad}")
print(f"Gold    : {gold!r}")
print(f"EM      : {compute_em(context_bad, gold)}")

## 7. How F1 is Computed in This Project

**Token-level F1:** tokenise both the retrieved context and the gold answer by whitespace,
then compute precision/recall/F1 on the multiset token overlap.

- **Precision** = overlapping tokens / total tokens in prediction
- **Recall**    = overlapping tokens / total tokens in gold
- **F1**        = 2 × P × R / (P + R)

In [ ]:
from collections import Counter

def compute_f1(prediction: str, gold: str) -> float:
    """Token-level F1 between prediction context and gold answer."""
    pred_tokens = prediction.lower().split()
    gold_tokens = gold.lower().split()

    if not pred_tokens or not gold_tokens:
        return 0.0

    pred_counter = Counter(pred_tokens)
    gold_counter = Counter(gold_tokens)
    overlap = sum((pred_counter & gold_counter).values())

    precision = overlap / len(pred_tokens)
    recall    = overlap / len(gold_tokens)

    if precision + recall == 0.0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


# Walk through a concrete example step by step
prediction = "the battle was fought near new york city in 1776"
gold = "new york"

pred_tokens = prediction.lower().split()
gold_tokens = gold.lower().split()

pred_counter = Counter(pred_tokens)
gold_counter = Counter(gold_tokens)
overlap = sum((pred_counter & gold_counter).values())

precision = overlap / len(pred_tokens)
recall    = overlap / len(gold_tokens)
f1        = 2 * precision * recall / (precision + recall)

print(f"Prediction : {prediction}")
print(f"Gold       : {gold}")
print(f"Pred tokens: {pred_tokens}")
print(f"Gold tokens: {gold_tokens}")
print(f"Overlap    : {overlap}  (shared tokens: {list((pred_counter & gold_counter).elements())})")
print(f"Precision  : {overlap}/{len(pred_tokens)} = {precision:.4f}")
print(f"Recall     : {overlap}/{len(gold_tokens)} = {recall:.4f}")
print(f"F1         : {f1:.4f}")

## 8. Live Demo: Run EM & F1 on Real HotpotQA Items

Simulate what happens in `run_all_systems.py`:
use the HotpotQA distractor context as a stand-in for retrieved context
and compute EM/F1 to see whether the answer is reachable.

In [ ]:
print(f"{'#':<4} {'Answer':<30} {'EM':>4} {'F1':>6}  Question")
print("-" * 90)

for i, item in enumerate(items):
    gold = item['answer']
    # Flatten the HotpotQA distractor context into one string
    ctx_sentences = item['context']['sentences']
    context = ' '.join(sent for page_sents in ctx_sentences for sent in page_sents)

    em = compute_em(context, gold)
    f1 = compute_f1(context, gold)
    q_preview = item['question'][:50]
    print(f"{i:<4} {gold:<30} {em:>4.0f} {f1:>6.3f}  {q_preview}...")

## 9. Why Multi-Hop Questions Are Hard

Retrieve only the **first** supporting page and check if the answer is still there.

In [ ]:
print("Single-page retrieval simulation (only first supporting page):\n")
print(f"{'#':<4} {'Answer':<30} {'EM_full':>7} {'EM_1page':>8}  Question")
print("-" * 95)

for i, item in enumerate(items):
    gold = item['answer']
    sf_titles = item['supporting_facts']['title']

    # Full context (both supporting pages)
    ctx_titles = item['context']['title']
    ctx_sentences = item['context']['sentences']
    full_context = ' '.join(sent for page_sents in ctx_sentences for sent in page_sents)

    # Single-page context: only the FIRST supporting page
    first_title = sf_titles[0] if sf_titles else None
    single_page_context = ""
    if first_title and first_title in ctx_titles:
        idx = ctx_titles.index(first_title)
        single_page_context = ' '.join(ctx_sentences[idx])

    em_full  = compute_em(full_context, gold)
    em_1page = compute_em(single_page_context, gold)

    # Mark items where 1-page retrieval fails but full context succeeds
    flag = " <-- needs 2nd hop" if em_full == 1.0 and em_1page == 0.0 else ""
    q_preview = item['question'][:45]
    print(f"{i:<4} {gold:<30} {em_full:>7.0f} {em_1page:>8.0f}  {q_preview}...{flag}")